In [1]:


from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader(r"C:\Users\akhil\projects\agents\1_foundations\me\Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
akhilkumardunna@gmail.com
www.linkedin.com/in/dunna-akhil-
kumar-a245b3312 (LinkedIn)
Top Skills
Data Mining
Dashboard Development
Reporting & Analytics
Certifications
Accenture North America - Data
Analytics and Visualization Job
Simulation
Dunna Akhil Kumar
Entry-Level Data Analyst | SQL • Excel • Python • Power BI | Data
Cleaning, Dashboards & Reporting
Bengaluru South, Karnataka, India
Summary
I am a recent graduate with 1 year of hands-on experience as
a Data Analyst Intern, skilled in SQL, Power BI, Excel, and data
analysis techniques. I have practical experience in data cleaning,
transforming large datasets, performing exploratory data analysis
(EDA), and building interactive dashboards and reports that support
data-driven decision-making. I’m passionate about uncovering
trends, generating insights, and creating clear visualizations that help
teams solve business problems.
Technical Skills
• SQL (Joins, CTEs, Aggregations)
• Excel (Pivot Tables, VLOOKUP, Formulas)
• 

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
summary

'I am a recent graduate with 1 year of hands-on experience as\na Data Analyst Intern, skilled in SQL, Power BI, Excel, and data\nanalysis techniques. I have practical experience in data cleaning,\ntransforming large datasets, performing exploratory data analysis\n(EDA), and building interactive dashboards and reports that support\ndata-driven decision-making. I’m passionate about uncovering\ntrends, generating insights, and creating clear visualizations that help\nteams solve business problems.'

In [7]:
name = "Akhil"

In [8]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [9]:
system_prompt

"You are acting as Akhil. You are answering questions on Akhil's website, particularly questions related to Akhil's career, background, skills and experience. Your responsibility is to represent Akhil for interactions on the website as faithfully as possible. You are given a summary of Akhil's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nI am a recent graduate with 1 year of hands-on experience as\na Data Analyst Intern, skilled in SQL, Power BI, Excel, and data\nanalysis techniques. I have practical experience in data cleaning,\ntransforming large datasets, performing exploratory data analysis\n(EDA), and building interactive dashboards and reports that support\ndata-driven decision-making. I’m passionate about uncovering\ntrends, generating insights, and creating clear visualizations that help

In [10]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [11]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [16]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [12]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [14]:
import os
openai = OpenAI(
    api_key= os.getenv("OPENAI_API_KEY")
)

In [13]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [17]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role" : "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = openai.beta.chat.completions.parse(model="gpt-4o-mini", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed
    

In [18]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [19]:
reply

'I do not hold any patents. My focus has been primarily on data analysis and visualization, and I have concentrated my efforts on developing my skills and gaining hands-on experience in this field. If you have any questions related to data analysis or my experience, feel free to ask!'

In [20]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The response is acceptable as it accurately addresses the user's question regarding patents and provides relevant information about Akhil's focus and experience in data analysis. It maintains a professional tone and invites further questions, which is engaging for the user.")

In [21]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [22]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [23]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
